In [0]:

from datetime import datetime, timezone, timedelta
import random
import uuid
from pyspark.sql import types as T

# Repeatable scenario choices; unique identifiers for each run.
rng = random.Random(42)
batch_id = uuid.uuid4().hex

number_of_orders = 30
base_time = datetime.now(timezone.utc) - timedelta(days=7)

output_root = (
    "/Volumes/fintech_lakehouse/bronze/raw_events/orders"
)

events = []
used_order_ids = set()


def add_event(
    order_id,
    customer_id,
    amount,
    event_type,
    status,
    sequence_number,
    event_time,
):
    events.append({
        "event_id": str(uuid.uuid4()),
        "order_id": order_id,
        "customer_id": customer_id,
        "event_type": event_type,
        "status": status,
        "amount": amount,
        "sequence_number": sequence_number,
        "event_time": event_time.isoformat(),
    })


for index in range(number_of_orders):
    # Positive identifier that fits in a signed Spark BIGINT.
    order_id = uuid.uuid4().int & ((1 << 63) - 1)

    while order_id == 0 or order_id in used_order_ids:
        order_id = uuid.uuid4().int & ((1 << 63) - 1)

    used_order_ids.add(order_id)

    customer_id = rng.randint(1000, 1010)
    amount = rng.randint(500, 100000) / 100

    created_at = (
        base_time
        + timedelta(days=index % 5, minutes=index * 3)
    )

    # Every order starts with a creation event.
    add_event(
        order_id, customer_id, amount,
        "INSERT", "CREATED", 1, created_at,
    )

    # Some orders remain CREATED.
    if index % 5 == 0:
        continue

    paid_at = created_at + timedelta(hours=2)

    add_event(
        order_id, customer_id, amount,
        "UPDATE", "PAID", 2, paid_at,
    )

    # Some orders remain PAID.
    if index % 5 == 1:
        continue

    final_at = created_at + timedelta(days=1)

    if index % 5 == 2:
        add_event(
            order_id, customer_id, amount,
            "CANCEL", "CANCELLED", 3, final_at,
        )
    else:
        add_event(
            order_id, customer_id, amount,
            "UPDATE", "SHIPPED", 3, final_at,
        )


# Arrival order differs from business-event order.
rng.shuffle(events)

schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("order_id", T.LongType(), False),
    T.StructField("customer_id", T.LongType(), False),
    T.StructField("event_type", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("amount", T.DoubleType(), False),
    T.StructField("sequence_number", T.LongType(), False),
    T.StructField("event_time", T.StringType(), False),
])

events_df = spark.createDataFrame(events, schema=schema)

output_path = f"{output_root}/batch_{batch_id}"

# A unique folder per run; fail rather than overwrite an existing batch.
events_df.coalesce(1).write.mode("append").json(output_path)

print(f"Batch: {batch_id}")
print(f"Orders generated: {number_of_orders}")
print(f"Events generated: {len(events)}")
print(f"Written to: {output_path}")

display(
    events_df.orderBy("order_id", "sequence_number")
)

display(
    events_df.groupBy("event_type", "status").count()
)